In [70]:
import xarray as xr
import numpy as np
import pandas as pd
import dask.array
from zarr.storage import FSStore
import fsspec

In [71]:
sample = xr.open_dataset("versioned_nwp_sample.nc")

In [72]:
template = xr.Dataset(None, coords=sample.coords)

In [73]:
for name in template._coord_names:
    if np.issubdtype(template[name].dtype, np.datetime64):
        template[name].encoding["units"] = "hours since 2023-01-01 00:00:00"
    if np.issubdtype(template[name].dtype, np.timedelta64):
        template[name].encoding["units"] = "hours"

In [74]:
horizon_idx = pd.TimedeltaIndex([pd.Timedelta(hours=i) for i in range(61)])

In [75]:
template = template.reindex(horizon=horizon_idx)

In [76]:
model_run_time_utc_idx = pd.date_range(start="2023-01-01T00:00:00", end="2026-01-01T00:00:00", freq=pd.Timedelta(hours=3))

In [77]:
template = template.reindex(model_run_time_utc=model_run_time_utc_idx)

In [78]:
shape = tuple(v for v in template.coords.sizes.values())

In [79]:
dims = tuple(k for k in template.dims.keys())

/tmp/ipykernel_26554/3042545308.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dims = tuple(k for k in template.dims.keys())


In [80]:
empty_array = dask.array.empty(shape=shape, chunks=[shape[0], 1, 1, shape[3], shape[4]])

In [81]:
for variable in sample.data_vars:
    template[variable] = (dims, empty_array)

In [ ]:
template

<xarray.Dataset> Size: 196TB
Dimensions:                            (altitude_m: 3,
                                        model_run_time_utc: 8769, horizon: 61,
                                        y: 1606, x: 1906)
Coordinates:
  * altitude_m                         (altitude_m) float64 24B 0.0 2.0 10.0
  * model_run_time_utc                 (model_run_time_utc) datetime64[ns] 70kB ...
  * horizon                            (horizon) timedelta64[ns] 488B 00:00:0...
    latitude                           (y, x) float64 24MB ...
    longitude                          (y, x) float64 24MB ...
Dimensions without coordinates: y, x
Data variables:
    temperature_K                      (altitude_m, model_run_time_utc, horizon, y, x) float64 39TB dask.array<chunksize=(3, 1, 1, 1606, 1906), meta=np.ndarray>
    wind_u_m_s                         (altitude_m, model_run_time_utc, horizon, y, x) float64 39TB dask.array<chunksize=(3, 1, 1, 1606, 1906), meta=np.ndarray>
    wind_v_m_s                         (altitude_m, model_run_time_utc, horizon, y, x) float64 39TB dask.array<chunksize=(3, 1, 1, 1606, 1906), meta=np.ndarray>
    accumulated_direct_radiation_J_m2  (altitude_m, model_run_time_utc, horizon, y, x) float64 39TB dask.array<chunksize=(3, 1, 1, 1606, 1906), meta=np.ndarray>
    accumulated_global_radiation_J_m2  (altitude_m, model_run_time_utc, horizon, y, x) float64 39TB dask.array<chunksize=(3, 1, 1, 1606, 1906), meta=np.ndarray>

In [2]:
fs = fsspec.filesystem(
    protocol="az",
    account_name="sapvforecastuch",
    anon=False,
)

In [3]:
remote_store = FSStore(
    url="az://data/versioned_nwp",
    fs=fs,
)

In [4]:
ds = xr.open_zarr(remote_store)

In [ ]:
for var in list(ds.variables.keys()):
    print(var, ds[var].chunks)

In [ ]:
template.to_zarr(
    store=remote_store,
    compute=False,
)